In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split


In [2]:
train = np.load("train.npz")
test = np.load("test.npz")

In [3]:
train_data = train["data"]
train_labels = train["labels"]
test_data = test["data"]

In [5]:
print("Full train data:", train_data.shape)
print("Full train labels:", train_labels.shape)
print("Test data:", test_data.shape)

Full train data: (60000, 784)
Full train labels: (60000,)
Test data: (15000, 784)


In [4]:
# Split the training data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    train_data,
    train_labels,
    test_size=0.2,
    random_state=1,
    stratify=train_labels,
)

In [7]:
print("Training split:", X_train.shape)
print("Validation split:", X_val.shape)

Training split: (48000, 784)
Validation split: (12000, 784)


In [5]:
def one_hot(y, num_classes=10):
    mat = np.eye(num_classes)[np.array(y).reshape(-1)]
    return mat

y = [0, 2, 1]
nb_classes = 3
print(one_hot(y, nb_classes))

[[1. 0. 0.]
 [0. 0. 1.]
 [0. 1. 0.]]


In [6]:
def softmax(z):
    z = z - np.max(z, axis=-1, keepdims=True)
    
    expsum = np.sum(np.exp(z), axis=-1, keepdims=True)
    return np.exp(z) / expsum

z = np.array([1.0, 2.0, 3.0])
print(softmax(z), np.sum(softmax(z)))

z = np.array([1000.0, 1001.0, 1002.0])
print(softmax(z), np.sum(softmax(z))) # hits overflow, needs z - max(z)

z = np.array([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0]
])
print(softmax(z), np.sum(softmax(z), axis=-1, keepdims=True)) # hits size mismatch, needs keepdims

[0.09003057 0.24472847 0.66524096] 0.9999999999999999
[0.09003057 0.24472847 0.66524096] 0.9999999999999999
[[0.09003057 0.24472847 0.66524096]
 [0.09003057 0.24472847 0.66524096]] [[1.]
 [1.]]


In [7]:
def forward(X, W, b):
    # prediction
    z = X @ W + b

    # activation
    y_hat = softmax(z)

    return y_hat

# X.shape = (48000, 784)
# W.shape = (784, 10)
# b.shape = (10, )
X = np.random.randn(48000, 784)
W = np.random.randn(784, 10) * np.sqrt(2 / 784) # He initialization
b = np.zeros(10)
print(forward(X, W, b)[:2])

[[0.03537068 0.0112195  0.13429023 0.0800504  0.37964558 0.00470451
  0.04478769 0.2446452  0.01053072 0.05475548]
 [0.03734096 0.07479515 0.03489342 0.09002771 0.17869304 0.18422478
  0.01148583 0.04461315 0.27587168 0.06805428]]


In [8]:
def cross_entropy(y_hat, y, eps=1e-15):
    m = y_hat.shape[0]
    y_hat = np.clip(y_hat, eps, 1 - eps)
    return -1/m * np.sum(y * np.log(y_hat))

y = one_hot([0, 2], num_classes=3)
y_hat = np.array([
    [0.7, 0.2, 0.1],  
    [0.1, 0.1, 0.8]   
])
print(cross_entropy(y_hat, y))

y_hat = np.array([
    [1.0, 0.0, 0.0],
    [0.0, 0.0, 1.0]
])
print(cross_entropy(y_hat, y)) # hits division by log 0, needs clipping

0.2899092476264711
9.992007221626415e-16


In [9]:
def backward(X, y_hat, y, W, b, vW, vb, lr, momentum=0.9):
    m = y_hat.shape[0]

    # dout = dL/dy_hat * dy_hat/dz
    dout = (y_hat - y) / m

    # dW = dL/dy_hat * dy_hat/dz * dz/dW = dout * X
    # X.shape = (48000, 748)
    # dout.shape = (48000, 10)
    dW = X.T @ dout

    # sum of dout across samples
    db = np.sum(dout, axis=0)

    # gradient descent
    vW = momentum * vW - lr * dW
    vb = momentum * vb - lr * db
    W += vW
    b += vb

    return W, b, vW, vb

X = np.array([[1.0, 2.0], [3.0, 4.0]])
y = one_hot([0, 1], num_classes=2)
W = np.zeros((2, 2))
b = np.zeros(2)
y_hat = np.array([[0.8, 0.2], [0.1, 0.9]])

#W_new, b_new = backward(X, y_hat, y, W, b, lr=0.1)
#print(W_new, b_new)

In [10]:
def predict(X, W, b):
    y_hat = forward(X, W, b)
    winner = np.argmax(y_hat, axis=-1)

    return winner

X = np.array([[1.0, 0.0], [0.0, 1.0]])
W = np.array([[10.0, -10.0], [-10.0, 10.0]])
b = np.zeros(2)
print(predict(X, W, b))

print(predict(X[0], W, b)) 

[0 1]
0


In [14]:
# setup
# X_train_norm = X_train / 255.0 if X_train.max() > 1.0 else X_train
# X_val_norm = X_val / 255.0 if X_val.max() > 1.0 else X_val
X_norm = train_data / 255.0
X_test_norm = test_data / 255.0

mean_px = np.mean(X_norm, axis=0)

X_norm = X_norm - mean_px
X_test_norm = X_test_norm - mean_px

# y_train_oh = one_hot(y_train, num_classes=10)
# y_val_oh = one_hot(y_val, num_classes=10)
y_oh = one_hot(train_labels, num_classes=10)

np.random.seed(42)
W = np.random.randn(784, 10) * np.sqrt(2 / 784) # He init
b = np.zeros(10)

# momentum
vW = np.zeros_like(W)
vb = np.zeros_like(b)
momentum = 0.9

# hyperparameters
epochs = 70
batch_size = 128
lr = 0.05
m = X_norm.shape[0]

In [312]:
# mini batch gsd
for epoch in range(epochs):
    indices = np.random.permutation(m)
    X_shuf = X_norm[indices]
    y_shuf = y_oh[indices]

    # loop over batches
    for i in range(0, m, batch_size):
        X_batch = X_shuf[i:i+batch_size]
        y_batch = y_shuf[i:i+batch_size]

        y_hat = forward(X_batch, W, b)
        W, b, vW, vb = backward(X_batch, y_hat, y_batch, W, b, vW, vb, lr, momentum)

    # log progrees
    if (epoch + 1) % 5 == 0 or epoch == 0:
        probs = forward(X_norm, W, b)
        loss = cross_entropy(probs, y_oh)
        acc = np.mean(predict(X_norm, W, b) == train_labels)
        print(f"Epoch {epoch + 1}/{epochs} | Train Loss: {loss:.4f} | Train Acc: {acc:.4f}")    

    lr *= 0.98

Epoch 1/70 | Train Loss: 0.3151 | Train Acc: 0.9101
Epoch 5/70 | Train Loss: 0.2701 | Train Acc: 0.9245
Epoch 10/70 | Train Loss: 0.2566 | Train Acc: 0.9289
Epoch 15/70 | Train Loss: 0.2511 | Train Acc: 0.9309
Epoch 20/70 | Train Loss: 0.2470 | Train Acc: 0.9324
Epoch 25/70 | Train Loss: 0.2445 | Train Acc: 0.9327
Epoch 30/70 | Train Loss: 0.2422 | Train Acc: 0.9335
Epoch 35/70 | Train Loss: 0.2405 | Train Acc: 0.9342
Epoch 40/70 | Train Loss: 0.2393 | Train Acc: 0.9345
Epoch 45/70 | Train Loss: 0.2383 | Train Acc: 0.9343
Epoch 50/70 | Train Loss: 0.2376 | Train Acc: 0.9344
Epoch 55/70 | Train Loss: 0.2367 | Train Acc: 0.9350
Epoch 60/70 | Train Loss: 0.2360 | Train Acc: 0.9352
Epoch 65/70 | Train Loss: 0.2355 | Train Acc: 0.9355
Epoch 70/70 | Train Loss: 0.2352 | Train Acc: 0.9356


In [15]:
# experiment grid
config = [
    {
        "name": "sgd",
        "lr": 0.1,
        "batch_size": 64,
        "momentum": 0.0,
        "decay": 1.0,
        "epochs": 50,
        "mean_center": False
    },
    {
        "name": "sgd_mean",
        "lr": 0.1,
        "batch_size": 64,
        "momentum": 0.0,
        "decay": 1.0,
        "epochs": 50,
        "mean_center": True
    },
    {
        "name": "momentum_128",
        "lr": 0.05,
        "batch_size": 128,
        "momentum": 0.9,
        "decay": 0.95,
        "epochs": 60,
        "mean_center": True
    },
    {
        "name": "momentum_slowlr",
        "lr": 0.03,
        "batch_size": 128,
        "momentum": 0.9,
        "decay": 0.98,
        "epochs": 60,
        "mean_center": True
    },
    {
        "name": "momentum_64",
        "lr": 0.03,
        "batch_size": 64,
        "momentum": 0.9,
        "decay": 0.97,
        "epochs": 50,
        "mean_center": True
    },

]

In [21]:
results = []

X_tr_raw, X_va_raw, y_tr, y_va = train_test_split(
    train_data, train_labels, test_size=0.2, random_state=1, stratify=train_labels
)

X_tr_base = X_tr_raw / 255.0
X_val_base = X_va_raw / 255.0
y_tr_oh = one_hot(y_tr, num_classes=10)
y_val_oh = one_hot(y_va, num_classes=10)
mean_vec = np.mean(X_tr_base, axis=0)

for cfg in config:
    print(f"Running {cfg['name']}")

    if cfg["mean_center"]:
        X_tr = X_tr_base - mean_vec
        X_val = X_val_base - mean_vec
    else:
        X_tr = X_tr_base
        X_val = X_val_base

    np.random.seed(42)
    W = np.random.randn(784, 10) * np.sqrt(2 / 784)
    b = np.zeros(10)
    vW = np.zeros_like(W)
    vb = np.zeros_like(b)

    lr = cfg['lr']
    batch_size = cfg["batch_size"]
    momentum = cfg["momentum"]
    decay = cfg["decay"]
    epochs = cfg["epochs"]
    m = X_tr.shape[0]

    best_val_acc = 0.0
    best_epoch = 0

    for epoch in range(epochs):
        indices = np.random.permutation(m)
        X_shuf = X_tr[indices]
        y_shuf = y_tr_oh[indices]

        for i in range(0, m, batch_size):
            X_b = X_shuf[i : i + batch_size]
            y_b = y_shuf[i : i + batch_size]

            y_hat = forward(X_b, W, b)
            W, b, vW, vb = backward(X_b, y_hat, y_b, W, b, vW, vb, lr, momentum)

        lr *= decay

        val_preds = predict(X_val, W, b)
        val_acc = np.mean(val_preds == y_va)
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch + 1

    final_tr_acc = np.mean(predict(X_tr, W, b) == y_tr)
    final_val_loss = cross_entropy(forward(X_val, W, b), y_val_oh)

    print(f"Peak Val Acc: {best_val_acc:.4f} (Epoch {best_epoch}) | Final Val Loss: {final_val_loss:.4f}")
    results.append({
        "Config": cfg["name"],
        "Batch": batch_size,
        "LR": cfg["lr"],
        "Momentum": momentum,
        "Decay": decay,
        "Peak Val Acc": round(best_val_acc, 4),
        "Peak Epoch": best_epoch,
        "Final Val Loss": round(final_val_loss, 4),
        "Final Train Acc": round(final_tr_acc, 4)
    })

import pandas as pd
summary_df = pd.DataFrame(results).sort_values(by="Peak Val Acc", ascending=False)
display(summary_df)

Running sgd
Peak Val Acc: 0.9217 (Epoch 48) | Final Val Loss: 0.2896
Running sgd_mean
Peak Val Acc: 0.9213 (Epoch 30) | Final Val Loss: 0.2888
Running momentum_128
Peak Val Acc: 0.9211 (Epoch 22) | Final Val Loss: 0.2880
Running momentum_slowlr
Peak Val Acc: 0.9213 (Epoch 36) | Final Val Loss: 0.2882
Running momentum_64
Peak Val Acc: 0.9214 (Epoch 14) | Final Val Loss: 0.2899


,Config,Batch,LR,Momentum,Decay,Peak Val Acc,Peak Epoch,Final Val Loss,Final Train Acc
0,sgd,64,0.10,0.0,1.00,0.9218,48,0.2896,0.9331
4,momentum_64,64,0.03,0.9,0.97,0.9214,14,0.2899,0.9361
1,sgd_mean,64,0.10,0.0,1.00,0.9213,30,0.2888,0.9337
3,momentum_slowlr,128,0.03,0.9,0.98,0.9212,36,0.2882,0.9341
2,momentum_128,128,0.05,0.9,0.95,0.9211,22,0.2880,0.9338


In [ ]:
# ==========================================================
# FINAL PUSH: Refined Gamma (0.78) + Pure Convex Descent (No L2) + Cosine LR (0.10) + 10 Seeds
# ==========================================================

gamma = 0.78  # fine-tuned contrast scaling (preserves stroke geometry, sharpens loops)
X_boost = (train_data / 255.0) ** gamma
X_test_boost = (test_data / 255.0) ** gamma

mean_boost = np.mean(X_boost, axis=0)
X_boost_norm = X_boost - mean_boost
X_test_boost_norm = X_test_boost - mean_boost

y_oh = one_hot(train_labels, num_classes=10)

# Cosine Annealing Schedule with 2-epoch Warmup
def get_lr_cosine(epoch, total_epochs, lr_max=0.10, lr_min=1e-4, warmup_epochs=2):
    if epoch < warmup_epochs:
        return lr_min + (lr_max - lr_min) * (epoch + 1) / warmup_epochs
    progress = (epoch - warmup_epochs) / (total_epochs - warmup_epochs)
    return lr_min + 0.5 * (lr_max - lr_min) * (1 + np.cos(np.pi * progress))

# Backward pass with Momentum and Pure Gradient Descent (no L2 to avoid underfitting)
def backward_opt(X, y_hat, y, W, b, vW, vb, lr, momentum=0.9):
    m = y_hat.shape[0]
    dout = (y_hat - y) / m
    dW = X.T @ dout
    db = np.sum(dout, axis=0)
    vW = momentum * vW + lr * dW
    vb = momentum * vb + lr * db
    W -= vW
    b -= vb
    return W, b, vW, vb

# 10 diverse random seeds to cancel mini-batch stochastic variance
seeds = [42, 101, 7, 999, 2026, 123, 777, 888, 314, 555]
W_list, b_list = [], []
m = X_boost_norm.shape[0]

epochs = 60
batch_size = 128
lr_max = 0.10

for s in seeds:
    print(f"Training seed {s} (Cosine LR 0.10, Gamma 0.78, Pure SGD)...")
    np.random.seed(s)
    W_seed = np.random.randn(784, 10) * np.sqrt(2 / 784)
    b_seed = np.zeros(10)
    vW = np.zeros_like(W_seed)
    vb = np.zeros_like(b_seed)

    for epoch in range(epochs):
        cur_lr = get_lr_cosine(epoch, epochs, lr_max=lr_max, lr_min=1e-4, warmup_epochs=2)
        idx = np.random.permutation(m)
        X_s = X_boost_norm[idx]
        y_s = y_oh[idx]

        for i in range(0, m, batch_size):
            X_b = X_s[i : i + batch_size]
            y_b = y_s[i : i + batch_size]
            y_hat = forward(X_b, W_seed, b_seed)
            W_seed, b_seed, vW, vb = backward_opt(X_b, y_hat, y_b, W_seed, b_seed, vW, vb, cur_lr, momentum=0.9)

    W_list.append(W_seed)
    b_list.append(b_seed)

# Average weights into a single Perceptron
W = np.mean(W_list, axis=0)
b = np.mean(b_list, axis=0)

train_acc = np.mean(predict(X_boost_norm, W, b) == train_labels)
print(f"\nFinal Averaged Perceptron Train Acc: {train_acc:.4f}")

# Generate final submission
predictions = predict(X_test_boost_norm, W, b)
submission = pd.DataFrame({"ID": range(len(predictions)), "target": predictions})
submission.to_csv("submission.csv", index=False)
submission.to_csv("submission_final.csv", index=False)
print("Saved submission.csv and submission_final.csv!")


In [ ]:
train_accuracy = np.mean(predict(X_boost_norm, W, b) == train_labels)
print(f"Training accuracy: {train_accuracy:.4f}")
predictions = predict(X_test_boost_norm, W, b)


In [317]:
submission = pd.DataFrame({
    "ID": range(len(predictions)),
    "target": predictions,
})

In [318]:
submission.to_csv("submission.csv", index=False)
print("Created submission.csv")

Created submission.csv
